# 条件 A/B/C 比較（QE 3.1 vs 3.5-flash-lite）

検索層 2×2 実験の per-question CSV を pandas/numpy で読み込んで比較する。

- `df_a` : 条件A dense+sparse（QEなし・3.1/3.5 共通）
- `df_b`, `df_c` : 条件B/C（QE = gemini-3.1-flash-lite）
- `df_b35`, `df_c35` : 条件B/C（QE = gemini-3.5-flash-lite）
- `df_base` : Base dense-only（scores_20260719.json の structure・参考）

列: id, question, match_type, source, difficulty, notation_variant, recall, strict_hit, reciprocal_rank, expected, retrieved, expanded_query

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

# repo ルート配下の data/eval を cwd から上方向に探す（notebook をどこで開いても効く）
EVAL = next(p / "data/eval" for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/eval").is_dir())
DATE = "20260721"


def load_csv(name: str) -> pd.DataFrame:
    df = pd.read_csv(EVAL / name)
    # strict_hit は 'True'/'False' 文字列で入るので 0/1 に正規化
    df["strict_hit"] = (df["strict_hit"].astype(str) == "True").astype(int)
    for c in ("recall", "reciprocal_rank"):
        df[c] = df[c].astype(float)
    return df


# ── それぞれ変数に格納 ──
df_a   = load_csv(f"scores_condition_a_{DATE}.csv")                       # A（QEなし）
df_b   = load_csv(f"scores_condition_b_{DATE}.csv")                       # B（QE 3.1）
df_c   = load_csv(f"scores_condition_c_{DATE}.csv")                       # C（QE 3.1）
df_b35 = load_csv(f"scores_condition_b_gemini-3.5-flash-lite_{DATE}.csv") # B（QE 3.5）
df_c35 = load_csv(f"scores_condition_c_gemini-3.5-flash-lite_{DATE}.csv") # C（QE 3.5）

# Base（dense-only）は JSON 側にあるので per_question を DataFrame 化
_base = json.loads((EVAL / "scores_20260719.json").read_text(encoding="utf-8"))["structure"]["per_question"]
df_base = pd.DataFrame(_base)
df_base["strict_hit"] = df_base["strict_hit"].astype(int)

CONDS = {
    "Base (dense)":        df_base,
    "A (dense+sparse)":    df_a,
    "B (QE 3.1)":          df_b,
    "C (both 3.1)":        df_c,
    "B (QE 3.5)":          df_b35,
    "C (both 3.5)":        df_c35,
}
{k: v.shape for k, v in CONDS.items()}

{'Base (dense)': (66, 11),
 'A (dense+sparse)': (66, 12),
 'B (QE 3.1)': (66, 12),
 'C (both 3.1)': (66, 12),
 'B (QE 3.5)': (66, 12),
 'C (both 3.5)': (66, 12)}

## 全体スコア（条件別）

In [2]:
def summarize(df: pd.DataFrame) -> pd.Series:
    return pd.Series({
        "n": len(df),
        "recall@5": df["recall"].mean(),
        "strict_hit": df["strict_hit"].mean(),
        "mrr": df["reciprocal_rank"].mean(),
    })


summary = pd.DataFrame({k: summarize(v) for k, v in CONDS.items()}).T
summary["n"] = summary["n"].astype(int)
summary.round(3)

,n,recall@5,strict_hit,mrr
Base (dense),66,0.598,0.500,0.562
A (dense+sparse),66,0.621,0.530,0.555
B (QE 3.1),66,0.636,0.500,0.614
C (both 3.1),66,0.674,0.545,0.615
B (QE 3.5),66,0.611,0.500,0.562
C (both 3.5),66,0.629,0.515,0.586


## match_type 別 recall@5（AND=多ソース比較の弱点が見える）

In [3]:
by_mt = pd.DataFrame({
    k: v.groupby("match_type")["recall"].mean() for k, v in CONDS.items()
})
by_mt.round(3)

,Base (dense),A (dense+sparse),B (QE 3.1),C (both 3.1),B (QE 3.5),C (both 3.5)
match_type,,,,,,
and,0.352,0.407,0.407,0.426,0.383,0.426
or,0.684,0.684,0.684,0.684,0.684,0.684
single,0.850,0.850,0.900,1.000,0.850,0.850


## QE モデル入替の per-question 差分（3.1 → 3.5）

同一 id で突き合わせ、`d = metric(3.5) - metric(3.1)`。負なら 3.5 で悪化。

In [4]:
def paired(df31: pd.DataFrame, df35: pd.DataFrame, metric: str = "recall") -> pd.DataFrame:
    m = df31[["id", "match_type", metric]].merge(
        df35[["id", metric]], on="id", suffixes=("_31", "_35"),
    )
    m["d"] = m[f"{metric}_35"] - m[f"{metric}_31"]
    return m


pC = paired(df_c, df_c35, "recall")
pB = paired(df_b, df_b35, "recall")

print("C recall  3.1→3.5:  mean_d =", round(pC["d"].mean(), 4),
      "| 改善", int((pC["d"] > 0).sum()), "/ 悪化", int((pC["d"] < 0).sum()),
      "/ 不変", int((pC["d"] == 0).sum()))
print("B recall  3.1→3.5:  mean_d =", round(pB["d"].mean(), 4),
      "| 改善", int((pB["d"] > 0).sum()), "/ 悪化", int((pB["d"] < 0).sum()),
      "/ 不変", int((pB["d"] == 0).sum()))

# 動いた質問だけ表示
pC[pC["d"] != 0].sort_values("d")[["id", "match_type", "recall_31", "recall_35", "d"]]

C recall  3.1→3.5:  mean_d = -0.0455 | 改善 3 / 悪化 6 / 不変 57
B recall  3.1→3.5:  mean_d = -0.0253 | 改善 2 / 悪化 5 / 不変 59


,id,match_type,recall_31,recall_35,d
6,old_07,single,1.0,0.0,-1.0
9,old_10,single,1.0,0.0,-1.0
17,old_18,single,1.0,0.0,-1.0
30,forum_73604,or,1.0,0.0,-1.0
60,generated_001,and,0.5,0.0,-0.5
65,generated_006,and,0.5,0.0,-0.5
29,forum_73612,and,0.0,0.5,0.5
42,forum_72094,and,0.5,1.0,0.5
22,forum_76903,or,0.0,1.0,1.0


## bootstrap 95%CI（任意の2条件のペア差）

`paired_diff.py` と同じ考え方を notebook 内で。任意の DataFrame ペアで再利用可。

In [5]:
def boot_ci(df_a: pd.DataFrame, df_b: pd.DataFrame, metric: str = "recall", n_boot: int = 10000, seed: int = 42):
    m = df_a[["id", metric]].merge(df_b[["id", metric]], on="id", suffixes=("_a", "_b"))
    d = (m[f"{metric}_b"] - m[f"{metric}_a"]).to_numpy()
    rng = np.random.default_rng(seed)
    boot = np.array([rng.choice(d, size=len(d), replace=True).mean() for _ in range(n_boot)])
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return {"n": len(d), "mean_diff": float(d.mean()), "ci95": (float(lo), float(hi)),
            "crosses_0": bool(lo <= 0 <= hi)}


# 例: C(3.1) vs Base、C(3.5) vs Base
print("C(3.1) vs Base:", boot_ci(df_base, df_c, "recall"))
print("C(3.5) vs Base:", boot_ci(df_base, df_c35, "recall"))
print("C 3.1 vs 3.5  :", boot_ci(df_c35, df_c, "recall"))

C(3.1) vs Base: {'n': 66, 'mean_diff': 0.0757560606060606, 'ci95': (-0.010103030303030301, 0.1616151515151515), 'crosses_0': True}
C(3.5) vs Base: {'n': 66, 'mean_diff': 0.030301515151515152, 'ci95': (-0.042930303030303034, 0.10605757575757577), 'crosses_0': True}
C 3.1 vs 3.5  : {'n': 66, 'mean_diff': 0.045454545454545456, 'ci95': (-0.022727272727272728, 0.12121212121212122), 'crosses_0': True}
